In [1]:
import networkx as nx
from rule_extender.utils import *
#from rule_extender.LPscoring import *
from pathlib import Path
import os

from rule_extender.lookforpath import lookforpath
import pandas as pd
from rdflib import URIRef #todo: maybe remove this
import networkx as nx

In [2]:
ROOT_DIR = Path(os.path.abspath('.')).resolve()

#todo: is to be parsed programmatically in the future
dataset_folder = ROOT_DIR / 'datasets'
dataset_name= 'NELL995'
rules_file= ROOT_DIR / "rule_mining" / dataset_name / "split_mined_rules-100"
train = dataset_folder / dataset_name / "NELL995_train.tsv"
valid =  dataset_folder / dataset_name /  "NELL995_valid.tsv"
test =   dataset_folder / dataset_name /  "NELL995_test.tsv"
schema_path = dataset_folder / dataset_name / "NELL.ontology.ttl"
temp_dir = ROOT_DIR / 'temp'
N = 3000


In [3]:
ontology = load_ontology(str(schema_path))
functional_properties = find_functional_prop(ontology)
dom_range_dict = find_dom_range(ontology) # <prop>:([domains],[ranges])
disjoints_dict = find_disjoint_classes(ontology) # <class>:[disjoint_classes]
rules, topN, pred_rule_index = parse_rules_file(str(rules_file), N)

graph_train = nx.MultiDiGraph()
with open(train, 'r') as rf:
    for line in rf.readlines():
        s, p, o = line.strip('\n').split('\t')
        graph_train.add_edge(s, o, key=p)

In [8]:
def generate_triple_predictions(kg: nx.MultiDiGraph, known_entity:str, target_entity: str, candidate_rules:list,
                                limit:int, mask_object:bool = True, functional:bool = False,
                                disjoint_req:list = list()):
    predictions = dict()
    unique_predictions = set()

    for conf,cand in candidate_rules:
        if len(unique_predictions) > limit:
            # rationale is that the rankings and prediction are accurate up to N, usually 100
            break
        all_valid_groundings = set() #this is the results of all possible grounding of the target variable
        open_variables = list(set([t[1] for t in cand] + [t[2] for t in cand])) #variables to be assigned
        if mask_object:
            counts = lookforpath(kg=kg,
                    remaining_rule=cand[1:],
                    target_var=cand[0][2],
                    last_assigned_variable=cand[0][1],
                    open_vars=[v for v in open_variables if v!= cand[0][1]],
                    grounded_vars={cand[0][1]:known_entity},
                    results_list= all_valid_groundings,
                    limit = limit, functional=functional, disjoint_req=disjoint_req)
        else: #the subject is being masked
            counts = lookforpath(kg=kg,
                    remaining_rule=cand[1:][::-1],
                    target_var=cand[0][1],
                    last_assigned_variable=cand[0][2],
                    open_vars=[v for v in open_variables if v!= cand[0][2]],
                    grounded_vars={cand[0][2]:known_entity},
                    results_list= all_valid_groundings,
                    limit = limit, functional=functional, disjoint_req=disjoint_req)
        if counts and len(all_valid_groundings) >0 : #counts == True if not exception triggered
            #update the list of predictions and the list of unique predictions
            if conf in predictions.keys():
                predictions[conf] = predictions[conf].union(all_valid_groundings)
            else:
                predictions[conf] = set(all_valid_groundings)
            unique_predictions = unique_predictions.union(all_valid_groundings)
        # build the ranking
        sorted_predictions = [pred for key in sorted(predictions.keys(), reverse=True) for pred in predictions[key]]
        # aggregate according to 'max rank' criterion: only consider the highest conf rule for each predicted target
        # return list(dict.fromkeys(sorted_predictions))
        if predictions is None:
            print()
        return predictions

# todo: something not quite right here, need to dobule check

def generate_predictions(train_kg, test_file, out_file,
                         functional_properties:list, pred_rules_index:dict, disjoint_dict:dict, domain_range_dict:dict,
                         limit:int = 100,debug=False):

    test_triples = pd.read_csv(test_file, sep= '\t', header = None, names = ['s','p','o'])
    if debug: test_triples = test_triples[:1000]
    with open(out_file, 'w') as of: #following the approach from anyburl
        for i, trip in test_triples.iterrows():
            p = trip.iloc[1]
            s = trip.iloc[0]
            o = trip.iloc[2]

            if URIRef('http://ste-lod-crew.fr/nell/ontology/') + p in functional_properties:
                functional = True
            else:
                functional = False
            if i % 5000 == 0:
                print(i)

            candidate_rules = pred_rules_index[p] if p in pred_rules_index.keys() else []
            # if functional and any(data == p for u,v,data in train_kg.out_edges(s,keys=True)):
            if functional:  # this checks if a (s,p,?) exists, which would except the functional rule
                outgoing = [data for u, v, data in train_kg.out_edges(s, keys=True)]
                if p in outgoing:
                    #todo: log occurrence
                    pass
            sorted_o_predictions = generate_triple_predictions(kg=train_kg, known_entity=s, target_entity=o,
                                                                      candidate_rules=candidate_rules, limit=limit,
                                                                      mask_object=True, functional=functional,
                                                                      disjoint_req=disjoint_dict[
                                                                          domain_range_dict[p][1]])

            sorted_s_predictions = generate_triple_predictions(kg = train_kg, known_entity = o, target_entity = s,
                                                                      candidate_rules = candidate_rules, limit = limit,
                                                                      mask_object=False, functional= functional,
                                                                      disjoint_req = disjoint_dict[domain_range_dict[p][0]])
            of.write(f'{s}\t{p}\t{o}\n')
            if sorted_o_predictions is None:
                print()
            # of.write('subjects: ' +  "".join(f"{s} \t {key} \t " for key, string_list in sorted_o_predictions.items() for s in string_list))



In [11]:
generate_predictions(train_kg=graph_train, test_file=test, out_file=str(temp_dir / 'anyburl_test_predictions.txt'),
                     functional_properties=functional_properties, pred_rules_index=pred_rule_index,
                     disjoint_dict=disjoints_dict, domain_range_dict=dom_range_dict, limit=100, debug=True)
print()

0








KeyboardInterrupt: 